# CSE 151B Starter Notebook

End-to-end pipeline:
1. Environment setup with `uv`
2. Load competition dataset
3. Inference with Qwen3-4B-Thinking via vLLM (INT8)
4. Score against ground truth
5. Save results to JSONL

`public.jsonl` has answers (measure accuracy locally). Private set has no answers — skip eval and submit raw responses.

## 1. Environment Setup

`uv` for package management. Install once, restart kernel.

### Comment out after first install.

In [1]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

# Create a virtual environment
!uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

'wget' is not recognized as an internal or external command,
operable program or batch file.
'uv' is not recognized as an internal or external command,
operable program or batch file.
'.venv' is not recognized as an internal or external command,
operable program or batch file.


Done. Restart the kernel before proceeding.
Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.


'.venv' is not recognized as an internal or external command,
operable program or batch file.


### Re-run each session to activate the venv.

In [2]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

'source' is not recognized as an internal or external command,
operable program or batch file.


## 2. Imports & Configuration

- `DATA_PATH` - public dataset
- `OUTPUT_PATH` - per-question results
- `GPU_ID` - device index
- `MAX_TOKENS` - generation cap

In [ ]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "../data/public.jsonl"
OUTPUT_PATH = "../results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

JSONL. Fields: `id`, `question`, `options` (MCQ only), `answer`.

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

Two system prompts (MCQ / free-form). `build_prompt()` returns the `(system, user)` pair.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician solving competition problems across algebra, "
    "calculus, statistics, probability, linear algebra, geometry, number theory, "
    "and discrete math. Reason carefully, then emit a final answer the automated "
    "grader can parse.\n"
    "\n"
    "# Method\n"
    "- Identify the problem type and the most direct solution method.\n"
    "- State the key formula or theorem before computing.\n"
    "- Keep every quantity in EXACT symbolic form throughout. Do not round, "
    "truncate, or convert to decimal at any intermediate step.\n"
    "- Verify by substitution, a limit/edge case, or solving a simpler instance "
    "before committing. If a check fails, restart from the broken step.\n"
    "\n"
    "# Final-answer format (STRICT — grader uses regex + sympy with 1e-8 relative tolerance)\n"
    "- Place the final answer inside \\boxed{...} at the very end of your response.\n"
    "- ALWAYS prefer exact symbolic forms over decimals. Decimals fail the 1e-8 "
    "tolerance whenever the gold answer is a fraction, radical, or transcendental.\n"
    "    Use \\boxed{\\frac{1}{3}}    NOT \\boxed{0.333}\n"
    "    Use \\boxed{\\sqrt{2}}      NOT \\boxed{1.414}\n"
    "    Use \\boxed{\\frac{\\pi}{4}} NOT \\boxed{0.7854}\n"
    "    Use \\boxed{\\ln 2}         NOT \\boxed{0.6931}\n"
    "    Use \\boxed{e^{2}}          NOT \\boxed{7.389}\n"
    "- Emit a decimal ONLY when (a) the exact value is itself a finite decimal "
    "(e.g. 2.5, 17), or (b) the problem explicitly asks for a decimal / a "
    "specific number of decimal places. Never write \"\\approx\" inside the box.\n"
    "- Inside \\boxed{}: bare value(s) only. No units, no \"x =\", no words, "
    "no \\text{...}.\n"
    "\n"
    "# Multiple sub-answers\n"
    "If the question asks for several values (e.g. \"Q1=[ANS]\\nQ3=[ANS]\\nIQR=[ANS]\"), "
    "put ALL of them in a SINGLE \\boxed{} separated by commas, in the order "
    "asked:\n"
    "    \\boxed{580, 660, 80}\n"
    "    \\boxed{\\frac{1}{2}, \\sqrt{3}, \\pi}\n"
    "Do not split sub-answers across multiple boxes."
)


SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician answering a multiple-choice competition "
    "problem. Solve rigorously, then output ONE letter.\n"
    "\n"
    "# Method\n"
    "- Compute the answer in EXACT symbolic form first (fractions, radicals, "
    "\\pi, e — no decimal rounding mid-calculation).\n"
    "- Only at the very end, convert to a decimal to match against the listed "
    "options (the options are typically rounded to 3 decimals).\n"
    "- If your exact value does not match any option within ordinary rounding, "
    "RECOMPUTE from scratch. Do not pick the visually closest option blindly.\n"
    "- Verify with a quick sanity check (sign, magnitude, units) before committing.\n"
    "\n"
    "# Final-answer format (STRICT)\n"
    "- Output exactly one \\boxed{X} at the very end, where X is a single "
    "capital letter (A, B, C, ...) matching your chosen option.\n"
    "- Examples: \\boxed{C}, \\boxed{E}.\n"
    "- No option text, no \"Option C\", no extra characters — just the letter."
)


# Qwen-team-recommended math suffix (used in MATH-500 evals).
USER_PROMPT_SUFFIX = (
    "\n\nPlease reason step by step, and put your final answer within \\boxed{}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(
            f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options)
        )
        user = f"{question}\n\nOptions:\n{opts_text}{USER_PROMPT_SUFFIX}"
        return SYSTEM_PROMPT_MCQ, user
    return SYSTEM_PROMPT_MATH, f"{question}{USER_PROMPT_SUFFIX}"



# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM (faster on a real GPU)

Qwen3-4B-Thinking-2507 with INT8 via BitsAndBytes.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.85,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

## 5. Load Model with Transformers (DataHub fallback)

Qwen3-4B-Thinking-2507 with INT4 via BitsAndBytes.

In [7]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


c:\Users\aniru\.conda\envs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0430 22:00:30.974000 27760 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]c:\Users\aniru\.conda\envs\ml\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards: 100%|██████████| 3/3 [00:03<00:00,  1.17s/it]


## 6. Generate Responses

Format with the chat template, then `llm.generate()` in one batched pass.

### Generate with vLLM

In [8]:
subset = data[:20]
prompts = []
for item in subset:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={subset[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

### Generate with Transformers (DataHub)

In [9]:
# responses = []
# subset = data[:20]
# print(f"Generating responses for {len(subset)} questions...")
# for idx, item in enumerate(tqdm(subset, desc="Generating")):
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#
#     inputs = tokenizer(
#         [prompt_text],
#         return_tensors="pt",
#         truncation=True,
#         max_length=16384,
#     ).to(llm.device)
#
#     with torch.no_grad():
#         output_ids = llm.generate(
#             **inputs,
#             max_new_tokens=MAX_TOKENS,
#             temperature=0.6,
#             top_p=0.95,
#             top_k=20,
#             repetition_penalty=1.0,
#             do_sample=True,
#         )
#
#     new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
#
# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 1126 questions...


Generating:   0%|          | 0/1126 [00:00<?, ?it/s]c:\Users\aniru\.conda\envs\ml\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Generating:   0%|          | 2/1126 [16:12<151:45:46, 486.07s/it]


KeyboardInterrupt: 

## 7. Score Responses

MCQ: extract boxed letter, exact match. Free-form: `Judger.auto_judge()`. Records: `{id, is_mcq, gold, response, correct}`.

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data[:len(responses)], responses), total=len(responses), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Accuracy by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

JSONL output. With eval: `{id, is_mcq, gold, response, correct}`. Without eval (private): `{id, is_mcq, response}`. Toggle via `SAVE_EVAL`.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

Some directions:
- prompt engineering (system prompts, few-shot)
- sampling parameters (temperature, top_p, majority voting)
- fine-tuning